Testing Purposes

In [1]:
!pip install numpy gdal earthpy matplotlib torch torchvision torchaudio scikit-learn opencv-python tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 53.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.2/22.2 MB 30.3 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
 

In [2]:
# Imports

import os
import random

import numpy as np


from osgeo import gdal, gdal_array
import earthpy.plot as ep

import matplotlib.pyplot as plt

import torch
import torch.optim as optim
import torch.nn as nn

from sklearn.model_selection import train_test_split

import cv2 as cv

import tqdm


In [4]:
from google.colab import drive
import shutil
import tarfile
import gdown

# # Mount Google Drive
# drive.mount('/content/drive')

# # Download the tar file from Google Drive using gdown
# file_id = '1-cU2qx7XY_lwCC7PKOnnNRkeyRto80gC'
# gdown.download(f'https://drive.usercontent.google.com/download?id={file_id}', 'dataset.tar', quiet=False)
# https://drive.usercontent.google.com/download?id=1-cU2qx7XY_lwCC7PKOnnNRkeyRto80gC

!curl 'https://drive.usercontent.google.com/download?id=1-cU2qx7XY_lwCC7PKOnnNRkeyRto80gC&confirm=t&uuid=7f38a5b7-15f7-49d1-a619-15644cdaefc0&at=APcmpowjuuMKU-AUFeNbDiKgHP6Z%3A1745592082397' -H 'User-Agent: Mozilla/5.0 (X11; Linux x86_64; rv:136.0) Gecko/20100101 Firefox/136.0' -H 'Accept: text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8' -H 'Accept-Language: en-US,en;q=0.5' -H 'Accept-Encoding: gzip, deflate, br, zstd' -H 'DNT: 1' -H 'Alt-Used: drive.usercontent.google.com' -H 'Connection: keep-alive' -H 'Referer: https://drive.usercontent.google.com/' -H 'Cookie: NID=523=EYb8H_Tcm4ZxTzO8PEJSE2gY-rX0B3IDCDjYvgiLqWb_PlOpn_9qfH3dq6X7P5su8V53TKe1osu4aAEPZQ9TSX07Tyw7qoJwHeQPsF6jFFaxPrXlProCpgohD3HKCXY6w3G-HNPPo_ZPHDcMdSZ4EIKSr6xVmQ5egHxWOtgUTcx0MEuqeg2VziGX3IywTBIXN3xByyagjtiIY3RLCBTOr06PXyN26aevHQKOB0IhHYeKoUU5DMRUeXqDLYyTeqtAah9sDTZUkhgVxyiCmKFlyDEP7eRsNboy8DbhI1MOHRkwnjwlFCgVo6l4bb46rZMtt7HXQvB5xyhHZLVwB2AIL_GSbTMD48eKUHf5TSNPrbtwLNh10XI0QKmE8yV0MPsIWr9_iHwk_b1SyzSsMDhPaHKzTkbvhAZEJFwHuM9yCq7gvk2t4HE9oy1NcAJ8TwOorkSKG6-CmcKFU_JkHuh64Ov-FAGQnm4UgAdiARmWLik3qXPgfhY_PKUTaGI2MEH0D1pCmEYatvkj-tn4xgaOtA_mWOor-ndYb33CSlWsUk4jdW0Lc8Ko6JhkVStENpykTHbCWyMqHB1AhUt-lAE0Nh7fyOb1Jd7b4pElyTalV55Sang884QRaDokNRSIYL-nbsIbcNq9COaoGztlLYM2OgIepqjqsSiz4cluBxzTmCKkIgnAXfGTjVGsqOzJTE4dDGB0VEMOYL_mudIRKrVgcXUP75X-CFbCDbnItk0FHXDkwXxwV0F9JgDaDw-mOh7nUaM_rBcIQkTMy_YX50V8xz-XiwpA5PT6TmtqgKTxrQT15-yMEsdeSnLS6Z0rNxHZQBA51Ft6EowyjqhB9gMlCGQgt_iJA9FATXnYsJ7ieYfWrX-j2BME8wfQj4hRSHmsoSO9ykEKWofIDmUKBzGbC_WcV5232UWJnWqw0hJbSf721aYHNqssPX6UL66HJvyEbPVmHjfnVsIBS_0YrEuAiWlY2HM03NDezmm2hCzdjOp3-12qfu8jOOYX87uggPPxOI2ug2ZWEfou-o_3sQ0DSi7dx6MwSiZhtaqCbR3rKlvyDpN5COaoodHNEdAz_cR-GUy4JDtQPrnsKqCq35fFyvPHD2qtUVGikKLlq8W4HLe4MqIflERI8ebrlSNOb3hr2M2vq9NRf6BflOceFzxXAAoee3ekZsO3ZjTPnH430Vobx9wcwrpJcK6QPKiusKVzr5gPmy6CPtNt6LBM7UzjZ7wrbNLO3IZttL4uiH9QMs2n6YVkzv-1_pHTV8UwxXqRYM-V4l3CT305IMqn4mwAFDoAAcYtb273-LzW__xGIA4feNeI5me-OG4B3GVIZLdlpSaYx_4D8IdHHa0uxiosANyPo4PGMJqhiWuRcUYE2NAHwU7XRjYQs42Cdnw0jCogr5IUmemSTMWiF9X2mDvG3Wdm80wah0noI864QfipACPbAxHs0mV1B6Gv0BdIThl7F-zWyUwK5iEPrW3Dex-vSUtBHMa1rIZh-cbCNpHG4n0Hg6f93ruQwZ8WBkilJP5NmDehgGsffKM_fG5hZmmJ8nrGrYmjlGdcRVbQZbOyKpQ0WfDWyn-OW86YPUrxTOU0Ow; SID=g.a000vwjK_8InjgLvfAEaZmy9rLNL2gUlf7EER9aqIyKLRVMUS49KZ8g8a_VEjkwqoI8tHFx87wACgYKASYSARcSFQHGX2Mi8oOYpME90PTDb61SoanpWhoVAUF8yKp8jTsYf7ARcQqBsysXErmD0076; __Secure-1PSID=g.a000vwjK_8InjgLvfAEaZmy9rLNL2gUlf7EER9aqIyKLRVMUS49K2VaURO01RgV1Ad3mKkntzwACgYKAYoSARcSFQHGX2Micd9l0BcHm-qnoHSrqESXNhoVAUF8yKoeG5l8ze6C_kAGa-aDazq20076; __Secure-3PSID=g.a000vwjK_8InjgLvfAEaZmy9rLNL2gUlf7EER9aqIyKLRVMUS49Kw1E4ImtDFDMDnrcywF3NMQACgYKAT0SARcSFQHGX2MixcX71UiXDx-Ilym4ExwInhoVAUF8yKqVi3-wUVFtxXseAoVZXzFW0076; HSID=APpnyt8sg3u7qMg_C; SSID=Amtk9dxUgN9a9raVD; APISID=jqSq_k-lKilkNuZD/A7Y1yKaHddqgTwCWI; SAPISID=qojQ1OOMtVMolJ5l/ArSWyHmxAw85NeGRx; __Secure-1PAPISID=qojQ1OOMtVMolJ5l/ArSWyHmxAw85NeGRx; __Secure-3PAPISID=qojQ1OOMtVMolJ5l/ArSWyHmxAw85NeGRx; SIDCC=AKEyXzUf_eKlWZdwXgcJTU3PbtwRZxHKhI8OrdY--JrSmtfTLirqKAi2zPYw5shsROdwcIh95AYb; __Secure-1PSIDCC=AKEyXzVVsr7XICFp6svgBxG-gAVgXIFxHrq42VJhZJivVJfUgox1f3oHPRa_ojWZgIukXQcytRg; __Secure-3PSIDCC=AKEyXzUjlQfAlCyVBhWSp3cukUzDm0jao59rCCBjA2HlPUbMAynEJFsDKQa3BXQgrYB8oSJSjynK; __Secure-1PSIDTS=sidts-CjIB7pHptSMsfKX5nH4Q03MsBwiqxO74KbQmJNCmn4z8dmFBoHMj8gB9ViaqJ8_fSNS9NxAA; __Secure-3PSIDTS=sidts-CjIB7pHptSMsfKX5nH4Q03MsBwiqxO74KbQmJNCmn4z8dmFBoHMj8gB9ViaqJ8_fSNS9NxAA; OSID=g.a000vAjK_w_ZbIFFo2sYJ4Nv1qZ0o13zg-YiUplqyIVZt531dVLWch9oCdLT-Hz96VjvRCnzoQACgYKAY4SARcSFQHGX2MifEK0Vtirlch9rLKeylg_ixoVAUF8yKrNemXb0QQuLO6Rvyhx4z4x0076; __Secure-OSID=g.a000vAjK_w_ZbIFFo2sYJ4Nv1qZ0o13zg-YiUplqyIVZt531dVLWPT3CutRPhjDi98AbUqfVlAACgYKAc0SARcSFQHGX2Mi9pWGlqlyRBO9itrU39GziRoVAUF8yKrtJVI3gexT1pijpwqm3dRg0076; AEC=AVcja2fZFVs888VZSLXgWcbJrhH0sEZBAP7SG4QrBexesQu0nglo1T0oDEk' -H 'Upgrade-Insecure-Requests: 1' -H 'Sec-Fetch-Dest: document' -H 'Sec-Fetch-Mode: navigate' -H 'Sec-Fetch-Site: cross-site' -H 'Sec-Fetch-User: ?1' -H 'Priority: u=0, i' -H 'Pragma: no-cache' -H 'Cache-Control: no-cache' -H 'TE: trailers' --output dataset.tar

# Extract the tar file
with tarfile.open('dataset.tar', 'r') as tar:
    tar.extractall(path='/content')  # Specify folder name to extract to

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 23.2G  100 23.2G    0     0  51.3M      0  0:07:44  0:07:44 --:--:-- 64.3M


In [5]:
content_dir = os.path.join("content", "train")
image_base_dir = os.path.join(content_dir, "data")
mask_base_dir = os.path.join(content_dir, "masks")

# Check if image directory exists
if not os.path.exists(image_base_dir):
    raise FileNotFoundError(f"Data Doesn't Exist: {image_base_dir}")

# List and sort files to maintain consistent ordering
image_paths = sorted([
    os.path.join(image_base_dir, fname)
    for fname in os.listdir(image_base_dir) if fname.endswith(".tif")
])

mask_paths = sorted([
    os.path.join(mask_base_dir, fname)
    for fname in os.listdir(mask_base_dir) if fname.endswith(".tif")
])

# Ensure matching counts
assert len(image_paths) == len(mask_paths), "Mismatch between image and mask counts"

# Optional: shuffle deterministically
combined = list(zip(image_paths, mask_paths))
random.seed(42)
random.shuffle(combined)
image_paths, mask_paths = zip(*combined)

# Split 60% train, 20% val, 20% test
X_temp, X_test, y_temp, y_test = train_test_split(image_paths, mask_paths, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42)

# Now you have:
print(f"Train: {len(X_train)} images")
print(f"Validation: {len(X_val)} images")
print(f"Test: {len(X_test)} images")

Train: 6343 images
Validation: 2115 images
Test: 2115 images


In [6]:
def convert_to_nparr(dataset):
    dtype = gdal_array.GDALTypeCodeToNumericTypeCode(dataset.GetRasterBand(1).DataType)
    arr = np.zeros((dataset.RasterYSize, dataset.RasterXSize, dataset.RasterCount), dtype=dtype)
    bands = []
    for i in range(dataset.RasterCount):
        arr[:, :, i] = dataset.GetRasterBand(i + 1).ReadAsArray()
        bands.append(arr[:, :, i])
    bands = np.stack(bands)
    return bands

In [ ]:
# ===================================== VISUALIZATION =====================================

# # Set a seed for reproducibility
# seed = 0
# random.seed(seed)

# # Select 25 random images
# random_images = random.sample(satellite_images, 1)



# for i, image_path in enumerate(random_images):
#     base_image = os.path.basename(image_path)
#     mask_path = os.path.join(mask_base_dir, base_image)

#     satellite_image = gdal.Open(image_path)
#     mask_image = gdal.Open(mask_path)

#     satellite_image = convert_to_nparr(satellite_image)
#     mask_image = convert_to_nparr(mask_image)

#     print(f"Image shape: {satellite_image.shape}")  # (bands, height, width)

#     # Also works with earthpy
#     ep.plot_rgb(satellite_image, rgb=(3, 2, 1), title=f"{image_path}")


In [15]:
# Data Loaders

class SatelliteDataset(torch.utils.data.Dataset):
    def __init__(self, image_paths, mask_paths, transform=None):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        mask_path = self.mask_paths[idx]

        image = gdal.Open(image_path)
        mask = gdal.Open(mask_path)

        if image is None:
            raise ValueError(f"Failed to load image at {image_path}")
        if mask is None:
            raise ValueError(f"Failed to load mask at {mask_path}")

        image = convert_to_nparr(image)
        mask = convert_to_nparr(mask)

        image = torch.from_numpy(image).float()
        mask = torch.from_numpy(mask).float()

        if self.transform:
            image = self.transform(image)
            mask = self.transform(mask)

        return image, mask



In [8]:
# Architecture
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, dropout=False, batchnorm=True, dropout_prob=0.5):
        super(ConvBlock, self).__init__()
        layers = [
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels) if batchnorm else nn.Identity(),
            nn.ReLU(inplace=True),
        ]
        layers += [
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels) if batchnorm else nn.Identity(),
            nn.ReLU(inplace=True),
        ]
        if dropout:
            layers.append(nn.Dropout(dropout_prob))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)



class UNet(nn.Module):
    in_channels = 4
    def prepare_image(self, x):
        return x
    def __init__(self, out_channels, dropout_prob=0.5):
        super(UNet, self).__init__()

        self.enc1 = ConvBlock(self.in_channels, 32)
        self.pool1 = nn.MaxPool2d(2) # 256x256

        self.enc2 = ConvBlock(32, 64)
        self.pool2 = nn.MaxPool2d(2) # 128x128

        self.enc3 = ConvBlock(64, 128)
        self.pool3 = nn.MaxPool2d(2) # 64x64

        self.enc4 = ConvBlock(128, 256)
        self.pool4 = nn.MaxPool2d(2) # 32x32

        self.enc5 = ConvBlock(256, 512)
        self.pool5 = nn.AvgPool2d(2) # 16x16

        self.bottleneck = ConvBlock(512, 1024)

        self.upconv5 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.dec5 = ConvBlock(1024, 512, batchnorm=False)

        self.upconv4 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.dec4 = ConvBlock(512, 256, batchnorm=False)

        self.upconv3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec3 = ConvBlock(256, 128, batchnorm=False)

        self.upconv2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec2 = ConvBlock(128, 64, batchnorm=False)

        self.upconv1 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.dec1 = ConvBlock(64, 32, dropout=True, batchnorm=False, dropout_prob=dropout_prob)

        self.out_conv = nn.Conv2d(32, out_channels, kernel_size=1)

        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        x = self.prepare_image(x)

        enc1 = self.enc1(x)
        enc1_pool = self.pool1(enc1)

        enc2 = self.enc2(enc1_pool)
        enc2_pool = self.pool2(enc2)

        enc3 = self.enc3(enc2_pool)
        enc3_pool = self.pool3(enc3)

        enc4 = self.enc4(enc3_pool)
        enc4_pool = self.pool4(enc4)

        enc5 = self.enc5(enc4_pool)
        enc5_pool = self.pool5(enc5)

        bottleneck = self.bottleneck(enc5_pool)

        dec5 = self.upconv5(bottleneck)
        dec5 = torch.cat((dec5, enc5), dim=1)
        dec5 = self.dec5(dec5)

        dec4 = self.upconv4(dec5)
        dec4 = torch.cat((dec4, enc4), dim=1)
        dec4 = self.dec4(dec4)

        dec3 = self.upconv3(dec4)
        dec3 = torch.cat((dec3, enc3), dim=1)
        dec3 = self.dec3(dec3)

        dec2 = self.upconv2(dec3)
        dec2 = torch.cat((dec2, enc2), dim=1)
        dec2 = self.dec2(dec2)

        dec1 = self.upconv1(dec2)
        dec1 = torch.cat((dec1, enc1), dim=1)
        dec1 = self.dec1(dec1)

        dec0 = self.out_conv(dec1)
        out = self.sigmoid(dec0)
        return out

In [12]:
def dice_coefficient(pred_mask: torch.Tensor, true_mask: torch.Tensor, eps=1e-6) -> float:
    intersection = (pred_mask * true_mask).sum()
    total_pixels = pred_mask.sum() + true_mask.sum()
    dice = (2.0 * intersection + eps) / (total_pixels + eps)
    return dice.item()

In [11]:
# Hyperparameters

BATCH_SIZE = 16
DROPOUT_PROB = 0.1
LEARNING_RATE = 0.001
NUM_EPOCHS = 10
CRITERION = nn.CrossEntropyLoss()


In [16]:
# Data Loaders

from torchvision import transforms

# Define the transform for training images
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(degrees=45),
])

train_dataset = SatelliteDataset(X_train, y_train, transform=train_transform)
val_dataset = SatelliteDataset(X_val, y_val)
test_dataset = SatelliteDataset(X_test, y_test)

train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = torch.utils.data.DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = torch.utils.data.DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = UNet(out_channels=1, dropout_prob=DROPOUT_PROB).to(device)  # 1 output channel for binary mask

criterion = CRITERION
optimizer = optim.SGD(model.parameters(), lr=LEARNING_RATE)

In [ ]:

for epoch in range(NUM_EPOCHS):
    model.train()
    train_loss = 0.0

    for img, mask in tqdm.tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}"):
        # print(f"Image shape: {img.shape}")  # (batch_size, bands, height, width)
        # print(f"Mask shape: {mask.shape}")

        img = img.to(device)
        mask = mask.to(device)

        # Forward pass
        outputs = model(img)
        loss = criterion(outputs, mask)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    # Validation (optional, but good practice)
    model.eval()
    total_dice = 0.0
    with torch.no_grad():
        for val_image, val_mask in val_loader:
            val_image, val_mask = val_image.to(device), val_mask.to(device)
            pred_mask = model(val_image)

            pred_mask = (pred_mask > 0.5).float()
            val_mask = (val_mask > 0.5).float()

            total_dice += dice_coefficient(pred_mask, val_mask)



    val_loss = total_dice / len(val_loader)

    print(
        f"Epoch [{epoch+1}/{NUM_EPOCHS}] - Train Loss: {train_loss:.4f} - Val Acc: {val_loss:.4f}")

Epoch 1/10: 100%|██████████| 397/397 [14:22<00:00,  2.17s/it]


Epoch [1/10] - Train Loss: 0.0000 - Val Acc: 0.7714


Epoch 2/10: 100%|██████████| 397/397 [14:23<00:00,  2.18s/it]


Epoch [2/10] - Train Loss: 0.0000 - Val Acc: 0.7714


Epoch 3/10: 100%|██████████| 397/397 [14:36<00:00,  2.21s/it]


...

Epoch 4/10: 100%|██████████| 397/397 [14:39<00:00,  2.21s/it]


Epoch [4/10] - Train Loss: 0.0000 - Val Acc: 0.7714


Epoch 5/10: 100%|██████████| 397/397 [14:38<00:00,  2.21s/it]


Epoch [5/10] - Train Loss: 0.0000 - Val Acc: 0.7714


Epoch 6/10:  19%|█▉        | 76/397 [02:49<11:41,  2.19s/it]

In [ ]:
# Save both the model's state_dict and the full model
torch.save(model.state_dict(), 'model_state_dict.pth')  # Recommended way
torch.save(model, 'model_full.pth')                     # Full model (less portable)

In [ ]:
# Load the model
model = UNet(out_channels=1)
model.load_state_dict(torch.load('model_state_dict.pth'))
model.eval()  # Set the model to evaluation mode
# Test the model
model.to("cuda")
total_dice = 0.0
with torch.no_grad():
    for test_images, test_masks in test_loader:
        test_images, test_masks = test_images.to(device), test_masks.to(device)

        # print(test_images.shape)

        pred_masks = model(test_images)

        pred_masks = (pred_masks > 0.5).float()
        test_masks = (test_masks > 0.5).float()

        total_dice += dice_coefficient(pred_masks, test_masks)

    test_dice = total_dice / len(test_loader)
    print(f"Test Dice Coefficient: {test_dice:.4f}")

Test Dice Coefficient: 0.8849


In [ ]:
from evaluate_pref import profile

# Load the model
model = UNet(out_channels=1)
model.load_state_dict(torch.load('model_state_dict.pth'))

num_ops, num_params = profile(model, (16, 4, 512, 512))

print(f"Number of operations: {num_ops}")
print(f"Number of parameters: {num_params}")

Not implemented for  ConvTranspose2d(512, 256, kernel_size=(2, 2), stride=(2, 2))
Not implemented for  Identity()
Not implemented for  Identity()
Not implemented for  ConvTranspose2d(256, 128, kernel_size=(2, 2), stride=(2, 2))
Not implemented for  Identity()
Not implemented for  Identity()
Not implemented for  ConvTranspose2d(128, 64, kernel_size=(2, 2), stride=(2, 2))
Not implemented for  Identity()
Not implemented for  Identity()
Not implemented for  ConvTranspose2d(64, 32, kernel_size=(2, 2), stride=(2, 2))
Not implemented for  Identity()
Not implemented for  Identity()
Not implemented for  Sigmoid()
Number of operations: tensor([1.4809e+12])
Number of parameters: tensor([7764353.])
